In [2]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client('scaler-sql-476312')

***If we retain the entre table with your agg function then we will use windows functions***

# **FOPO - Function() over(partition by , order by )**

Function() -  means you can use any of the SCAMM aggregations

In [3]:
# Here we need to know the total salary in the employees table
query = """
SELECT  sum(salary) AS total_salary
   FROM `HR_Data.employees`
"""
df = client.query(query).to_dataframe()
df

# But you can't see the all the data in employees table eventhough if you gave * symbol in select clause that asks you aggregate it so you can only see one val in employee table that is total_salary

,total_salary
0,692400


# **Uses of over()**

In [4]:
# Here we will use Over() to see all the data in the table including the total_salary
query = """
SELECT *, sum(salary) OVER() AS total_salary
FROM `HR_Data.employees`
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,total_salary
0,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,692400
1,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,692400
2,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,692400
3,103,Alexander,Hunold,AHUNOLD,590.423.4567,1990-01-03,IT_PROG,9000,NULL,102,60,692400
4,104,Bruce,Ernst,BERNST,590.423.4568,1991-05-21,IT_PROG,6000,NULL,103,60,692400
...,...,...,...,...,...,...,...,...,...,...,...,...
102,202,Pat,Fay,PFAY,603.123.6666,1997-08-17,MK_REP,6000,NULL,201,20,692400
103,203,Susan,Mavris,SMAVRIS,515.123.7777,1994-06-07,HR_REP,6500,NULL,101,40,692400
104,204,Hermann,Baer,HBAER,515.123.8888,1994-06-07,PR_REP,10000,NULL,101,70,692400
105,205,Shelley,Higgins,SHIGGINS,515.123.8080,1994-06-07,AC_MGR,12000,NULL,101,110,692400


In [5]:
# Find the Overall avg salary by using over window function
query = """
SELECT *, avg(salary) OVER() AS overall_avg_salary
FROM `HR_Data.employees`
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,overall_avg_salary
0,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,6471.028037
1,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,6471.028037
2,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,6471.028037
3,103,Alexander,Hunold,AHUNOLD,590.423.4567,1990-01-03,IT_PROG,9000,NULL,102,60,6471.028037
4,104,Bruce,Ernst,BERNST,590.423.4568,1991-05-21,IT_PROG,6000,NULL,103,60,6471.028037
...,...,...,...,...,...,...,...,...,...,...,...,...
102,202,Pat,Fay,PFAY,603.123.6666,1997-08-17,MK_REP,6000,NULL,201,20,6471.028037
103,203,Susan,Mavris,SMAVRIS,515.123.7777,1994-06-07,HR_REP,6500,NULL,101,40,6471.028037
104,204,Hermann,Baer,HBAER,515.123.8888,1994-06-07,PR_REP,10000,NULL,101,70,6471.028037
105,205,Shelley,Higgins,SHIGGINS,515.123.8080,1994-06-07,AC_MGR,12000,NULL,101,110,6471.028037


In [6]:
# Find the avg salary by department wise by using sub-queries
query = """
SELECT e.*, f.avg_sal
FROM `HR_Data.employees` e
INNER JOIN
  (
    SELECT department_id, avg(salary) AS avg_sal
    FROM `HR_Data.employees`
    GROUP BY 1
  ) f
  ON
    e.department_id = f.department_id
"""
df = client.query(query).to_dataframe()
df

# Next Cell we will find by using window function over()

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,avg_sal
0,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,19666.666667
1,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,19666.666667
2,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,19666.666667
3,103,Alexander,Hunold,AHUNOLD,590.423.4567,1990-01-03,IT_PROG,9000,NULL,102,60,5760.000000
4,104,Bruce,Ernst,BERNST,590.423.4568,1991-05-21,IT_PROG,6000,NULL,103,60,5760.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
102,202,Pat,Fay,PFAY,603.123.6666,1997-08-17,MK_REP,6000,NULL,201,20,9500.000000
103,203,Susan,Mavris,SMAVRIS,515.123.7777,1994-06-07,HR_REP,6500,NULL,101,40,6500.000000
104,204,Hermann,Baer,HBAER,515.123.8888,1994-06-07,PR_REP,10000,NULL,101,70,10000.000000
105,205,Shelley,Higgins,SHIGGINS,515.123.8080,1994-06-07,AC_MGR,12000,NULL,101,110,10150.000000


In [7]:
# Find the avg salary by department wise by using window function
query = """
SELECT *, avg(salary) OVER (PARTITION BY department_id) AS avg_salary
FROM `HR_Data.employees`
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,avg_salary
0,200,Jennifer,Whalen,JWHALEN,515.123.4444,1987-09-17,AD_ASST,4400,NULL,101,10,4400.000000
1,108,Nancy,Greenberg,NGREENBE,515.124.4569,1994-08-17,FI_MGR,12000,NULL,101,100,8600.000000
2,109,Daniel,Faviet,DFAVIET,515.124.4169,1994-08-16,FI_ACCOUNT,9000,NULL,108,100,8600.000000
3,110,John,Chen,JCHEN,515.124.4269,1997-09-28,FI_ACCOUNT,8200,NULL,108,100,8600.000000
4,111,Ismael,Sciarra,ISCIARRA,515.124.4369,1997-09-30,FI_ACCOUNT,7700,NULL,108,100,8600.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
102,170,Tayler,Fox,TFOX,011.44.1343.729268,1998-01-24,SA_REP,9600,0.2,148,80,8955.882353
103,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,19666.666667
104,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,19666.666667
105,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,19666.666667


In [9]:
# Display the details of employees having salaries greater than the average salary of all the employees
query = """
SELECT *
FROM
  (SELECT *, avg(salary) OVER () AS overall_avg_salary FROM `HR_Data.employees`) O
WHERE salary > overall_avg_salary
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,overall_avg_salary
0,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,6471.028037
1,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,6471.028037
2,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,6471.028037
3,103,Alexander,Hunold,AHUNOLD,590.423.4567,1990-01-03,IT_PROG,9000,NULL,102,60,6471.028037
4,108,Nancy,Greenberg,NGREENBE,515.124.4569,1994-08-17,FI_MGR,12000,NULL,101,100,6471.028037
5,109,Daniel,Faviet,DFAVIET,515.124.4169,1994-08-16,FI_ACCOUNT,9000,NULL,108,100,6471.028037
6,110,John,Chen,JCHEN,515.124.4269,1997-09-28,FI_ACCOUNT,8200,NULL,108,100,6471.028037
7,111,Ismael,Sciarra,ISCIARRA,515.124.4369,1997-09-30,FI_ACCOUNT,7700,NULL,108,100,6471.028037
8,112,Jose Manuel,Urman,JMURMAN,515.124.4469,1998-03-07,FI_ACCOUNT,7800,NULL,108,100,6471.028037
9,113,Luis,Popp,LPOPP,515.124.4567,1999-12-07,FI_ACCOUNT,6900,NULL,108,100,6471.028037


In [10]:
# Display the details of employees having salaries greater than the average salary of all employees in their respective department.
query = """
SELECT *
FROM
  (
    SELECT *, avg(salary) OVER (PARTITION BY department_id) AS dep_avg_salary
    FROM `HR_Data.employees`
  ) O
WHERE salary > dep_avg_salary
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,dep_avg_salary
0,108,Nancy,Greenberg,NGREENBE,515.124.4569,1994-08-17,FI_MGR,12000,NULL,101,100,8600.000000
1,109,Daniel,Faviet,DFAVIET,515.124.4169,1994-08-16,FI_ACCOUNT,9000,NULL,108,100,8600.000000
2,205,Shelley,Higgins,SHIGGINS,515.123.8080,1994-06-07,AC_MGR,12000,NULL,101,110,10150.000000
3,201,Michael,Hartstein,MHARTSTE,515.123.5555,1996-02-17,MK_MAN,13000,NULL,100,20,9500.000000
4,114,Den,Raphaely,DRAPHEAL,515.127.4561,1994-12-07,PU_MAN,11000,NULL,100,30,4150.000000
5,120,Matthew,Weiss,MWEISS,650.123.1234,1996-07-18,ST_MAN,8000,NULL,100,50,3475.555556
6,121,Adam,Fripp,AFRIPP,650.123.2234,1997-04-10,ST_MAN,8200,NULL,100,50,3475.555556
7,122,Payam,Kaufling,PKAUFLIN,650.123.3234,1995-05-01,ST_MAN,7900,NULL,100,50,3475.555556
8,123,Shanta,Vollman,SVOLLMAN,650.123.4234,1997-10-10,ST_MAN,6500,NULL,100,50,3475.555556
9,124,Kevin,Mourgos,KMOURGOS,650.123.5234,1999-11-16,ST_MAN,5800,NULL,100,50,3475.555556


# **Row Number**


For row number it cannot skip the numbers and it has duplicates of numbers

**In row_number() we can't give any arguments everything will be control on over() clause**

In [11]:
# Find the top 5 earning employees in the company
query = """
SELECT *, row_number() OVER (ORDER BY salary DESC) AS rnum
FROM `HR_Data.employees`
LIMIT 5
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,rnum
0,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,1
1,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,2
2,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,3
3,145,John,Russell,JRUSSEL,011.44.1344.429268,1996-10-01,SA_MAN,14000,0.4,100,80,4
4,146,Karen,Partners,KPARTNER,011.44.1344.467268,1997-01-05,SA_MAN,13500,0.3,100,80,5


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [12]:
# Find the top 5 earning employees in the company where there row_number<=5
query = """
SELECT *
FROM
  (
    SELECT *, row_number() OVER (ORDER BY salary DESC) AS rnum
    FROM `HR_Data.employees`
  ) r
WHERE rnum <= 5
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,rnum
0,100,Steven,King,SKING,515.123.4567,1987-06-17,AD_PRES,25000,NULL,NULL,90,1
1,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,2
2,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,3
3,145,John,Russell,JRUSSEL,011.44.1344.429268,1996-10-01,SA_MAN,14000,0.4,100,80,4
4,146,Karen,Partners,KPARTNER,011.44.1344.467268,1997-01-05,SA_MAN,13500,0.3,100,80,5


# **Rank and Dense_rank**


Rank - It can skip the numbers of rank and it has duplicates number of rank

Dense_rank - It cannot skip the numbers of dense rank and it has duplicates number of dense rank


In the ranking function we will mostly use order by

i.e - fopo - function() over(partition by , order by)

**In ranking functions we can't give any arguments everything will be control on over() clause**

In [13]:
# Fetch the details of employees with the 2nd highest salaries from each department.
query = """
SELECT *
FROM
  (
    SELECT
      *,
      dense_rank()
        OVER (PARTITION BY department_id ORDER BY salary DESC) AS sec_rank
    FROM `HR_Data.employees`
  ) r
WHERE sec_rank = 2
"""
df = client.query(query).to_dataframe()
df

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id,sec_rank
0,109,Daniel,Faviet,DFAVIET,515.124.4169,1994-08-16,FI_ACCOUNT,9000,NULL,108,100,2
1,206,William,Gietz,WGIETZ,51hr5.123.8181,1994-06-07,AC_ACCOUNT,8300,NULL,205,110,2
2,202,Pat,Fay,PFAY,603.123.6666,1997-08-17,MK_REP,6000,NULL,201,20,2
3,115,Alexander,Khoo,AKHOO,515.127.4562,1995-05-18,PU_CLERK,3100,NULL,114,30,2
4,120,Matthew,Weiss,MWEISS,650.123.1234,1996-07-18,ST_MAN,8000,NULL,100,50,2
5,104,Bruce,Ernst,BERNST,590.423.4568,1991-05-21,IT_PROG,6000,NULL,103,60,2
6,146,Karen,Partners,KPARTNER,011.44.1344.467268,1997-01-05,SA_MAN,13500,0.3,100,80,2
7,102,Lex,De Haan,LDEHAAN,515.123.4569,1993-01-13,AD_VP,17000,NULL,100,90,2
8,101,Neena,Kochhar,NKOCHHAR,515.123.4568,1989-09-21,AD_VP,17000,NULL,100,90,2
